<a href="https://colab.research.google.com/github/AllenInstitute/SPOCTT/blob/main/ABCAtlas_UseCases/SEAAD_Data_Access_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="title"></a>
# Seattle Alzheimer's Disease Brain Cell Atlas (SEA-AD) Data Access

This notebook is a **template**, not an analysis: it walks through browsing the SEA-AD single-cell data by brain region, picking a file, downloading it, and loading it into an AnnData object.

There's no analysis here on purpose. Copy this notebook and build on top of it once you've got the file you want loaded in.

It also references other types of data that are accessible

-----

Data source w/info: https://registry.opendata.aws/allen-sea-ad-atlas/


Direct data source: https://sea-ad-single-cell-profiling.s3.amazonaws.com/index.html


Publication: Gabitto, Travaglini, et al., *Nature Neuroscience* 2024: https://www.nature.com/articles/s41593-024-01774-5



---



<a name="audience"></a>
# Target audience for this notebook

This notebook is a starting point for **anyone** who wants to get SEA-AD single-cell data out of the public AWS bucket and into Python, whatever your level of experience.

**If you're new to Python or Colab:** you don't need to write any code. Start with the [quick intro to Colab](#colab-intro) below, then run each cell in order (click the ▶ button or press `Shift + Enter`), and only change the lines marked `**EDIT**`. Each step explains what it does and checks your choices before moving on.

Some cells (like the loading code in Section 4.3) are more advanced; you don't need to follow them line by line, just run them.

**If you're experienced:** skip the explanations and use it as a quick, working recipe for browsing the bucket, downloading a file with anonymous S3 access, and loading the metadata and embeddings without reading the full expression matrix into memory. Copy the cells you need into your own pipeline.

**What you'll end up with:**
- A data file downloaded from the bucket, with the option to save a copy to your Google Drive or computer ([Part 6](#sec-6))
- An AnnData object with cell metadata (`obs`), embeddings like UMAP (`obsm`) and extra information (`uns`), ready for your own analysis. New to AnnData? See [Section 4.2](#sec-4-2)

**What this notebook doesn't do:** any analysis. It also doesn't teach single-cell methods or Alzheimer's biology; see [About the Dataset](#intro-dataset) and the linked publication for that. It covers the RNAseq files; other SEA-AD data types are listed under [Additional Resources & Data](#add-resources).

**What you need:**
- A Google account (to run Colab and, optionally, save to Google Drive)
- No AWS account or credentials, because the data is public
- No software to install



---



<a name="colab-intro"></a>
# Intro to Colab



Welcome to Google Colab! This is the notebook environment we will use to access the data.

</br>
<img src="https://miro.medium.com/max/986/1*pimj8lXWwZnqLs2xVCV2Aw.png" width=300>
</br>
</br>


## What is Colab?
Colab is a browser-based environment - you can develop, run, and share code directly on your browser without having to install any software or libraries on your computer. Additionally, code written and run from colab runs on google's cloud computers, so you aren't limited by your laptop's speed or compute power. 📶

</br>


Colab can be especially helpful for data science tasks, because it gives you a space to write code, generate/view graphs, and organize your thought process.  For those of you who have used Jupyter before, colab acts a simple browser-based Jupyter notebook. We'll be using colab with Python, but colab can be used with a variety of different coding languages.

</br>

A Colab notebook (a file like this one) consists of cells, snippets of code or text that you can create, edit, and move around to keep things organized. These cells can be:

*   **text cells** 🔠
<br>or<br>
*   **code cells** 💻

You can **double click** <img src='https://miro.medium.com/v2/resize:fit:280/0*oa0XcvM99Y5clDsj.png' alt='mouse cursor images' height='25px'>
on a cell in order
to edit it.
</br>

----
## 0.1 Text Cells
<a name="text"></a>



This is a **text cell**. You can **double-click** to edit this cell. Text cells
use markdown syntax. To learn more, see the [markdown
guide](/notebooks/markdown_guide.ipynb).

You can format text cells by **changing** `the` *font*,  adding:
1.   Numbered lists, or
*   Bullet points


You can even add HTML to your text cells to embed images or add other fun effects.


<img src = "https://freesvg.org/img/icon-html.png" width=50>


You can also add math to text cells using [LaTeX](http://www.latex-project.org/)
to be rendered by [MathJax](https://www.mathjax.org). Just place the statement
within a pair of **\$** signs. For example `$\sqrt{3x-1}+(1+x)^2$` becomes
$\sqrt{3x-1}+(1+x)^2.$ <br><br>
To insert a text cell, click on a cell on the page and then click the "+ Text" button at the top. Or scroll to the middle of the notebook page before/after a cell and you should see these options appear: <br>
<img src="https://github.com/tmckim/python-crash-course/blob/main/imgs/ColabAddCodeTextImg.png?raw=1"/>


This will make a text cell directly above or below the cell you are on.


---
## 0.2 Code Cells
<a name="code-cells"></a>

Colab notebooks also consist of **coding cells**, cells that have executable code written in them.

To run a coding cell, either:
1. hover over the cell and click the "play" ▶ arrow that appears on the left upper corner
2. Or click into the cell and then press Ctrl+Enter (or Shift+Return).

You should see a check mark if the cell runs successfully ✔.

Coding cells can look like the following, where you can see raw code and comments (anything after the `#hashtag`). <br>

In [ ]:
parts = 6
# This is a comment! When there is a # before a line in a coding cell,
# python will ignore that line of code, rather than trying to run it.

'''
You can also
write longer
comments
by using
this syntax.
'''

print("Number of sections in this notebook:", parts)

In [ ]:
statement = "Hello, World!"
print(statement)

Additional resources to practice with Colab notebooks and get you started if you want even more before diving into the notebook below:

* [Welcome to Colab](https://colab.research.google.com/notebooks/intro.ipynb)
* [Colab Overview](https://colab.research.google.com/notebooks/basic_features_overview.ipynb)
* [External Data](https://colab.research.google.com/notebooks/io.ipynb)


---
<a name="save"></a>
# Before you start - Save this notebook!


**<font color="red" size=4> IMPORTANT: This notebook is READ-ONLY. To edit and run this notebook, follow the steps below.</font>**

When you open a new Colab notebook from the WebCampus (like you hopefully did for this one), you cannot save changes. So it's  best to store the Colab notebook in your personal drive `"File > Save a copy in Drive..."` **before** you do anything else.

The file will open in a new tab in your web browser, and it is automatically named something like: "**Copy of SEAAD_Data_Access_Colab.ipynb**". <br>You can rename this to just the title "**SEAAD_Data_Access_Colab.pynb**".<br>
Make sure you do keep an informative name so that you know what the file is.


___

**Where does the notebook get saved in Google Drive?**

By default, the notebook will be copied to a folder called `Colab Notebooks` at the root (home directory) of your Google Drive.

___
___


<a name="intro-dataset"></a>
# About the Dataset

- This dataset comes from the Seattle Alzheimer's Disease Brain Cell Atlas (SEA-AD), described in [Gabitto, Travaglini, et al. (2024, *Nature Neuroscience*)](https://www.nature.com/articles/s41593-024-01774-5).
- It contains ~1.4 million nuclei, the expression level of ~36,600 genes of 84 donors spanning the full range of Alzheimer's pathology, from no/low pathology to severe disease
- Nuclei are annotated into 139 molecular cell types using a reference taxonomy from the BRAIN Initiative
    - Includes neuronal and non-neuronal populations
- The aim of the study is to identify which brain cell types are most vulnerable to AD and at what stage of disease progression
- Findings revealed an early phase marked by loss of somatostatin-positive inhibitory neurons and a later phase marked by excitatory neuron loss

Video overviews:
- Full webinar (~1.5 hr): https://www.youtube.com/watch?v=lcS3o1N22Bc
- Webinar overview (~7 mins): https://www.youtube.com/watch?v=QON_qTo06oM


----

In [ ]:
#@title Allen Institute SEA-AD Webinar Overview (~7 mins) {display-mode: "form"}

from IPython.display import HTML

HTML("""
<iframe width="800" height="450"
        src="https://www.youtube.com/embed/QON_qTo06oM"
        title="YouTube video player"
        frameborder="0"
        referrerpolicy="strict-origin-when-cross-origin"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
        allowfullscreen>
</iframe>
""")

-----

<a name="add-resources"></a>
# Additional Resources & Data


This notebook contains specifics to accessing the RNAseq data.

- ATACseq (epigenetic) data is also available for MTG in the same folder path with single cell profiling

You can access other types of data associated with this paper and the study:

- The full list of data available: https://brain-map.org/consortia/sea-ad/our-data

Data available from AWS download (which would be most similar to this notebooks):
- Quantitative neuropathology (full resolution images, processed images, and quantifications): https://sea-ad-quantitative-neuropathology.s3.amazonaws.com/index.html
- Spatial transcriptomics: https://sea-ad-spatial-transcriptomics.s3.amazonaws.com/index.html

---
<a name="part-1"></a>
# Part 1: Setup


<a name="sec-1-1"></a>
## 1.1 Install & import packages


This is deliberately a smaller set of packages than a full analysis notebook. These will allow you to browse the bucket, download a file, and open it as an AnnData object.

In [ ]:
# Install required packages

!pip install anndata h5py boto3 s3fs "pandas==2.2.3" -q          # current pandas version compatible w/Colab. Can remove if upgrades, etc.

In [ ]:
# Import packages
import os

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm
import s3fs

# Colab's local disk (not Google Drive), which is storage tied to this session
# Not saved once the session ends, but doesn't use any of your personal Drive space
DATA_DIR = '/content'
os.makedirs(DATA_DIR, exist_ok = True)

BUCKET = 'sea-ad-single-cell-profiling'
fs = s3fs.S3FileSystem(anon = True)   # anonymous access since the bucket is public, no AWS credentials needed

---
<a name="part-2"></a>
# Part 2: Browse Available Data


<a name="sec-2-1"></a>
## 2.1 Review & pick a brain region

Single-cell data in this bucket is organized into one top-level folder per brain region. Run the cell below to see which regions are available.

In [ ]:
# list every top-level folder in the bucket; each one is a brain region
# (or a related grouping, e.g. Multiregion_2026)
entries = fs.ls(BUCKET + '/', detail = True)

regions = []
for e in entries:
    if e['type'] == 'directory':
        regions.append(e['name'].rstrip('/').split('/')[-1])

print("Available regions:")
for r in regions:
    print(' ', r)

Build one table of every file across every region/modality, so  sizes are available before picking anything below. This only goes one level into each modality folder.

Note: It does not go into nested subfolders like `donors_objects/` or `previous_objects/`, which hold per-donor files or older versions


In [ ]:
# File list
all_file_info = []

for region in regions:                                                          # regions list from the cell above
    region_path = f'{BUCKET}/{region}/'
    region_entries = fs.ls(region_path, detail = True)

    modality_names = []
    for e in region_entries:
        if e['type'] == 'directory':
            modality_names.append(e['name'].rstrip('/').split('/')[-1])

    for modality in modality_names:
        modality_path = f'{BUCKET}/{region}/{modality}/'
        file_entries = fs.ls(modality_path, detail = True)

        for e in file_entries:
            if e['type'] == 'file':
                all_file_info.append({
                    'region': region,
                    'modality': modality,
                    'name': e['name'].rstrip('/').split('/')[-1],
                    'size_GB': round(e['size'] / (1024**3), 2),
                    'full_path': e['name'],
                })

all_files_df = pd.DataFrame(all_file_info).sort_values(['region', 'modality', 'size_GB'], ascending = [True, True, False])
all_files_df

To see all the brain regions, options in the table, and so text isn't cut off when you want to view the information, set the viewing options in `pandas`

In [ ]:
pd.set_option('display.max_rows', None)       # show every row instead of truncating the middle
pd.set_option('display.max_colwidth', None)   # show full file names instead of cutting them off with "..."

all_files_df = pd.DataFrame(all_file_info).sort_values(['region', 'modality', 'size_GB'], ascending = [True, True, False])
all_files_df

Now you can pick a brain region you would like to obtain data for.

Ensure the text matches **exactly** what is displayed above in the `region` column

In [ ]:
# set this to one of the region names printed above
REGION = 'AnG'   # **EDIT** e.g. 'MTG', 'HIP', 'DLPFC', ...

assert REGION in regions, f"{REGION!r} isn't in the list above. Check spelling/capitalization"
print(f"Using region: {REGION}")

<a name="sec-2-2"></a>
## 2.2 Pick a data modality

Within a region, data is split further by modality (e.g. `RNAseq`, `ATACseq`). Not every region has every modality.

In [ ]:
# list the modality folders available within the chosen region
region_path = f'{BUCKET}/{REGION}/'
entries = fs.ls(region_path, detail = True)

modalities = []
for e in entries:
    if e['type'] == 'directory':
        modalities.append(e['name'].rstrip('/').split('/')[-1])

print(f"Modalities available under {REGION}:")
for m in modalities:
    print(' ', m)

Now choose the modality, editing the text to match **exactly** as displayed above

In [ ]:
# set this to one of the modality names printed above
MODALITY = 'RNAseq'  # **EDIT** # e.g. 'RNAseq', 'ATACseq'

assert MODALITY in modalities, f"{MODALITY!r} isn't in the list above. Check spelling/capitalization"
print(f"Using modality: {MODALITY}")

<a name="sec-2-3"></a>
## 2.3 Pick a file

Now list the files directly inside that region/modality folder (not the nested subfolders, like `donors_objects/` or `previous_objects/`, which usually hold per-donor files or older file versions) and choose one to download.

In [ ]:
# list files (not subfolders) directly under <REGION>/<MODALITY>/
modality_path = f'{BUCKET}/{REGION}/{MODALITY}/'
entries = fs.ls(modality_path, detail = True)

file_info = []
for e in entries:
    if e['type'] == 'file':
        file_info.append({
            'name': e['name'].rstrip('/').split('/')[-1],
            'size_GB': round(e['size'] / (1024**3), 2),
            'full_path': e['name'],
        })

files_df = pd.DataFrame(file_info).sort_values('size_GB', ascending = False)
files_df

**Note:**
- `all-nuclei` files include every nucleus that passed initial processing
- `final-nuclei` files are the same data after final filtering to remove low quality nuclei (this is
usually the one you want)

Files also come in a few formats (`.h5ad`, `.csv`, etc.). This notebook loads `.h5ad` files, so choose one of those.

Again, edit the filename to match **exactly** from above (you can copy and paste)

In [ ]:
# set this to one of the file names printed in the table above (must end in .h5ad)
FILENAME = 'SEAAD_AnG_RNAseq_final-nuclei.2026-06-22.h5ad'            # **EDIT**

assert FILENAME in files_df['name'].values, f"{FILENAME!r} isn't in the list above. Check spelling/capitalization"
assert FILENAME.endswith('.h5ad'), "this notebook loads .h5ad files. Pick one of those from the table above"

KEY = f'{REGION}/{MODALITY}/{FILENAME}'                              # full path within the bucket
LOCAL_PATH = os.path.join(DATA_DIR, FILENAME)                        # where it will be located locally

print(f"Selected: {KEY}")

---
<a name="part-3"></a>
# Part 3: Download the File

These files are large (several GB to tens of GB, depending on region and file type), so the code below uses `boto3` with a live progress bar. The cell below skips the download automatically if the file already exists at `LOCAL_PATH`, which was specified in the cell right above this text.
- This is the default google temporary working space under `Files` in the left side bar (click on the 📁 folder icon)
- The path is: content/

In [ ]:
# Set up an anonymous S3 client (the bucket is public, no AWS credentials needed)
s3 = boto3.client('s3', config = Config(signature_version = UNSIGNED))

print(f"Data will be stored at: {LOCAL_PATH}")

if os.path.exists(LOCAL_PATH):
    print(f"File already exists at {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH) / (1024**3):.2f} GB) - skipping download")
else:
    # Get the file size first, so the progress bar knows the total
    file_size = s3.head_object(Bucket = BUCKET, Key = KEY)['ContentLength']         # total size in bytes
    print(f"File size: {file_size / (1024**3):.2f} GB")

    # Download with a live progress bar
    with tqdm(total = file_size, unit = 'B', unit_scale = True, unit_divisor = 1024, desc = FILENAME) as pbar:
        def progress_callback(bytes_transferred):
            pbar.update(bytes_transferred)                                          # advance the bar by however many bytes just came in

        s3.download_file(BUCKET, KEY, LOCAL_PATH, Callback = progress_callback)

    print(f"\nDownload complete: {os.path.getsize(LOCAL_PATH) / (1024**3):.2f} GB")

---
<a name="part-4"></a>
# Part 4: Load the Data


<a name="sec-4-1"></a>
## 4.1 Look at the file's structure first


The file extension of the data file is `.h5ad`. <br>
This is a specialized HDF5-based format used in single-cell bioinformatics. Files are organized like a filesystem: groups (like folders) contain datasets (like files) or other groups


Before loading anything, it's worth seeing what's actually inside the file. Different files/regions won't necessarily have the exact same `obsm`/`uns` entries.

Here we loop over this structure to view the organization and what is contained in each

In [ ]:
# View the data structure using h5py
with h5py.File(LOCAL_PATH, 'r') as f:                                         # open the h5ad file directly in read-only mode
    for name in f.keys():                                                     # loop over the top-level keys (e.g. X, obs, var, obsm, layers, etc.)
        item = f[name]                                                        # get the object stored at this key
        if isinstance(item, h5py.Group):                                      # groups are like folders -- they contain nested datasets/subgroups
            print(f"{name}/  (group)")
            for sub_name in item.keys():
                print(f"    {sub_name}")
        else:
            print(f"{name}  shape = {item.shape}, dtype = {item.dtype}")      # if not a group, it's a dataset and can print shape and datatype

----

Info regarding data columns can be found in the supplemental material associated with the paper here: https://www.nature.com/articles/s41593-024-01774-5#Sec28

Some info also found in the `README.md` file with the data in the AWS bucket: https://sea-ad-single-cell-profiling.s3.amazonaws.com/README.md

---

<a name="sec-4-2"></a>
## 4.2 The AnnData ("Annotated Data") object

RNAseq data from this study, which contains ~1.4 million nuclei, the expression level of ~36,600 genes, plus donor demographics, QC metrics, and pre-computed coordinates, needs a container that keeps all of that aligned.

The standard for single-cell data in Python is **AnnData** (`adata`), from the [scverse](https://anndata.scverse.org/en/stable/) ecosystem:

<div align="center">
<img src="https://anndata.scverse.org/en/stable/_images/anndata_schema.svg" width="400">
</div>

**General explanation for AnnData**
- `X Matrix`: The central cell-by-feature (e.g., cell-by-gene) expression data matrix
- `Obs` / `Var`: Data frames storing metadata annotations for observations (cells) and variables (genes/features)
- `Obsp` / `Varp`: Pairwise relationships between observations or variables
  - Examples: k-nearest neighbors (KNN) connectivity graphs and cell/gene similarity matrices
- `Obsm` / `Varm`: Multi-dimensional annotations and embeddings
  - Examples: UMAP, t-SNE, and PCA coordinates
- `Layers`: Additional alternative count matrices, such as raw counts versus normalized data
- `Uns`: Unstructured metadata not tied to a specific dimensions
  - Examples: color palettes, clustering parameters, or analysis settings
---

**AnnData in the current file we are working with:**
| Piece | Shape | What it holds |
|---|---|---|
| `X` | cells × genes | The expression matrix itself, with the bulk of the data, stored as a sparse matrix |
| `layers['UMIs']` | cells × genes | Raw UMI counts, kept alongside the normalized `X` |
| `obs` | cells × metadata | Per-cell metadata: donor demographics, QC metrics, cell type calls. Rows line up 1:1 with `X` rows |
| `var` | genes × metadata | Per-gene metadata (`_index` = gene symbols, `gene_ids` = e.g. Ensembl IDs). Rows line up 1:1 with `X` columns |
| `obsm` | cells × k | Per-cell multi-dimensional arrays: `X_scVI` (latent embedding), `X_umap` (2D coordinates) |
| `obsp` | cells × cells | Cell-to-cell graphs: `distances` and `connectivities` from the nearest-neighbor graph |
| `uns` | — | Unstructured extras: color palettes, processing parameters, dataset title, etc. |
---

In scRNA-seq data, each row corresponds to a cell with a barcode, and each column corresponds to a gene with a gene id (cells × genes contained in `X`)

`obs` and `var` are the row/column labels for everything else

Once you know a cell's row index, that same index tells you its coordinates in `obsm['X_umap']`, its neighbors in `obsp`, and its expression values in that row of `X`.

<a name="sec-4-3"></a>
## 4.3 Load a lightweight AnnData object (no `X`)

`.X` is the large expression matrix, often tens of GB for these files. Opening it in backed mode (`ad.read_h5ad(LOCAL_PATH, backed = 'r')`) can crash in memory-constrained environments like Colab.

Since the goal here is just to get the data so it is accessible, this reads everything *except* `.X` directly via `h5py`:
- `obs` (cell metadata)
- every entry in `obsm` (precomputed embeddings, e.g. UMAP coordinates)
- every entry in `uns` (unstructured extras, e.g. color palettes)

It then wraps those in a lightweight AnnData object with the same `.obs[...]` / `.obsm[...]` / `.uns[...]` interface you'd get from a normal `ad.read_h5ad()` call, just without `.X` loaded into memory.

In [ ]:
# Load data into AnnData obj
try:
    from anndata.io import read_elem                                          # newer anndata versions
except ImportError:
    from anndata.experimental import read_elem                                # older anndata versions

with h5py.File(LOCAL_PATH, 'r') as f:
    obs_full = read_elem(f['obs'])                                            # full obs DataFrame
    n_vars = f['var'][f['var'].attrs['_index']].shape[0]                      # number of genes, from the var index length

    obsm_full = {}
    for key in f['obsm'].keys():                                              # read every entry in obsm, whatever it's called
        obsm_full[key] = read_elem(f['obsm'][key])

    uns_full = {}
    for key in f['uns'].keys():                                               # read every entry in uns, whatever it's called
        uns_full[key] = read_elem(f['uns'][key])

adata = ad.AnnData(
    obs = obs_full,
    obsm = obsm_full,
    uns = uns_full,
    shape = (obs_full.shape[0], n_vars),                                      # no X is loaded, shape is passed explicitly instead
)

print(adata)

<a name="sec-4-4"></a>
## 4.4 Quick data review

A few basic checks to confirm the data loaded as expected.

In [ ]:
print(f"Shape: {adata.n_obs} cells x {adata.n_vars} genes")

print("\nobs columns:")
for col in adata.obs.columns:
    print(' ', col)

print("\nobsm keys:", list(adata.obsm.keys()))
print("uns keys:", list(adata.uns.keys()))

adata.obs.head()

---
<a name="sec-5"></a>
# Part 5: Next steps

`adata` now has everything except `.X` loaded, which is enough for exploring cell metadata, subclass/supertype composition, QC columns, and precomputed embeddings.

If your next step needs actual expression values:
- **A subset of cells** (e.g. just one subclass): read `.X` in chunks directly via `h5py`, keeping only the rows you need. <mark>See [Section 3.2 of the SST analysis notebook](#) for a worked example of this pattern.
- **The whole matrix, and the file is small enough**: `ad.read_h5ad(LOCAL_PATH, backed = 'r')` may work fine for smaller files (e.g. single-region `final-nuclei` files under a few GB); it's the larger `all-nuclei`/multi-region files where this tends to crash in Colab.

---
<a name="sec-6"></a>
# Part 6: Saving the data outside of this Colab session

Anything in `/content` is **deleted when the runtime ends** (closing the tab, idle timeout, or *Runtime > Disconnect and delete runtime*). To keep the file, save it somewhere else before then.

There are several options for saving:

### <u>Option 1</u>: Download from the file browser
1. Click the 📁 **folder icon** in the left sidebar
2. Find the file (it's at the top level of `/content`). Click the refresh icon if it doesn't show up
3. Right-click the file (3 dots appear when hovering the mouse on the right side) and choose **Download**

### <u>Option 2</u>: Save a copy to your Google Drive (code below)
This option mounts your Drive, creates a folder, and copies the file there. You'll be asked to give Colab access to your Drive.

**Check your free space first.** These files are several GB, and free Google accounts have 15 GB in total.

### <u>Option 3</u>: Download the file directly from the AWS bucket
You can select the brain region folder and file from the website directly. Clicking on the name of a file will start a download locally to your computer: https://sea-ad-single-cell-profiling.s3.amazonaws.com/index.html


## 6.1 Save file in your Google Drive

The code below it set up to create a folder called: `SEAAD_data` where it will save to in your Google Drive

*Optional* : Edit the end of the line starting `DRIVE_FOLDER`. This will rename the folder where you would like to put the data.

This creates a new folder in your Google Drive with the corresponding name.





---


**Important**: The code below assumes you have already specified a file and `LOCAL_PATH` was defined in the last code cell of [Section 2.3 Pick a file](#sec-2-3).

It also needs to be downloaded in [Part 3](#part-3)


If you have not run that part yet, go back and do that first before coming here to run these cells


---



Run the code cell below.

A window will appear asking permission to give Colab access to your Google Drive to mount it and save the file there.

A progress bar will be displayed below the cell once the transfer starts.

In [ ]:
# Option 2: copy the file into a folder in your Google Drive
from google.colab import drive

# Check that a file was picked (Section 2.3) and downloaded (Part 3)
if 'LOCAL_PATH' not in globals():
    raise NameError("LOCAL_PATH isn't defined. Run the last code cell of Section 2.3 (Pick a file) first.")

if not os.path.exists(LOCAL_PATH):
    raise FileNotFoundError(
        f"No file at {LOCAL_PATH}.\n"
        "Run Part 3 (Download the File) first. If you already did, the runtime may have restarted and cleared /content, so run it again."
    )

drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/SEAAD_data'    # **EDIT** to change where it's saved
os.makedirs(DRIVE_FOLDER, exist_ok = True)

drive_path = os.path.join(DRIVE_FOLDER, FILENAME)
partial_path = drive_path + '.partial'                                   # copy here first, rename when complete

if os.path.exists(drive_path):
    print(f"Already in Drive - skipping copy:\n  {drive_path}")
else:
    file_size = os.path.getsize(LOCAL_PATH)
    chunk_size = 64 * 1024**2                                            # copy 64 MB at a time

    with open(LOCAL_PATH, 'rb') as src, open(partial_path, 'wb') as dst, \
         tqdm(total = file_size, unit = 'B', unit_scale = True, unit_divisor = 1024, desc = 'Copying to Drive') as pbar:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:                                                # empty chunk means the end of the file
                break
            dst.write(chunk)
            pbar.update(len(chunk))                                      # advance the bar by the bytes just copied

    os.rename(partial_path, drive_path)                                  # only gets the real name once fully copied
    print(f"Saved to:\n  {drive_path}")

print(f"\nIn Google Drive, look in: My Drive > {os.path.relpath(DRIVE_FOLDER, '/content/drive/MyDrive')}")

Before closing this notebook or ending the runtime, double check the file has been fully saved. It may take a few seconds to refresh and be fully uploaded, so you should wait a few minutes before stopping to ensure you fully downloaded the data.

If it doesn't appear, you should also run the code cell below.

In [ ]:
# Run this before closing the notebook, to make sure everything is fully uploaded to Drive
drive.flush_and_unmount()
print("All changes saved to Drive. Safe to close the notebook.")

# Part 7: Viewing the Location of All Data

https://registry.opendata.aws/allen-sea-ad-atlas/


Each bucket in the SEA-AD registry on AWS holds a different data modality:
- **sea-ad-single-cell-profiling**: single-cell transcriptomics (snRNAseq) & epigenomics (ATACseq)
- **sea-ad-quantitative-neuropathology**: full-resolution & processed pathology images & quantifications
- **sea-ad-spatial-transcriptomics**: spatial transcriptomics data



In [ ]:
# See what types of data are available
buckets = [
    'sea-ad-single-cell-profiling',
    'sea-ad-quantitative-neuropathology',
    'sea-ad-spatial-transcriptomics',
]

fs = s3fs.S3FileSystem(anon = True)

all_info = []
for bucket in buckets:
    entries = fs.ls(bucket + '/', detail = True)                                # top-level only
    for e in entries:
        all_info.append({
            'bucket': bucket,
            'name': e['name'].rstrip('/').split('/')[-1],
            'type': e['type'],                                                  # 'file' or 'directory'
            'full_path': e['name'],
        })

pd.DataFrame(all_info).sort_values(['bucket', 'type', 'name'])

You can now adapt the code above (Parts 1 - 6) if you'd like to review and pull in other data files to work with in a notebook to analyze in Python

# Questions or Looking for Resources?

* Search and post on the community forum: https://community.brain-map.org/tag/sea-ad/95

# Credits

* Thanks to colleagues at the Allen Institute for
sharing resources and data from this project.
* This notebook was made as part of the SPOCTT (Scientific and Public Outreach of Cell Type Taxonomies) Initiative
* Created by: Theresa H. McKim, PhD (tmckim@unr.edu)
* Shared under CC BY 4.0: you may reuse and adapt this notebook with attribution.